# 03. FMA REAL 오디오 검사

매칭한 REAL 296곡의 파일 존재 여부, 크기, 디코딩과 길이를 확인한다. 목록의 ID와 실제 파일이 다르면 이후 REAL/FAKE 짝이 틀어지므로 여기서 먼저 걸러낸다.

In [21]:
from pathlib import Path
import subprocess
import shutil
import pandas as pd
import numpy as np

PROJECT_ROOT = Path("/Users/seungjae/Desktop/SNU/기계학습 & 딥러닝/project")

MAPPING_PATH = PROJECT_ROOT / "data/metadata/fma_real_mapping.csv"
AUDIO_DIR = PROJECT_ROOT / "data/raw/FMA/selected_30s"
REPORT_PATH = PROJECT_ROOT / "data/metadata/fma_real_audio_validation.csv"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("MAPPING_PATH:", MAPPING_PATH)
print("AUDIO_DIR   :", AUDIO_DIR)

PROJECT_ROOT: /Users/seungjae/Desktop/SNU/기계학습 & 딥러닝/project
MAPPING_PATH: /Users/seungjae/Desktop/SNU/기계학습 & 딥러닝/project/data/metadata/fma_real_mapping.csv
AUDIO_DIR   : /Users/seungjae/Desktop/SNU/기계학습 & 딥러닝/project/data/raw/FMA/selected_30s


## 1. 매핑 파일과 다운로드된 MP3 개수 확인

`fma_real_mapping.csv`의 행 수와 실제 저장된 MP3 파일 수가 일치하는지 확인한다.

In [22]:
mapping = pd.read_csv(MAPPING_PATH)
audio_files = sorted(AUDIO_DIR.rglob("*.mp3"))

print("===== FMA REAL AUDIO CHECK =====")
print("Mapping rows      :", len(mapping))
print("Unique track_id   :", mapping["track_id"].nunique())
print("Downloaded MP3    :", len(audio_files))

===== FMA REAL AUDIO CHECK =====
Mapping rows      : 296
Unique track_id   : 296
Downloaded MP3    : 296


## 2. Track ID 일치 여부 확인

매핑 파일에서 기대되는 `track_id`와 실제 파일명의 `track_id`를 비교한다.

- `Missing = 0`
- `Extra = 0`

이면 정상이다.

In [23]:
# Track ID 일치 여부 확인
expected_ids = set(mapping["track_id"].astype(int).tolist())
downloaded_ids = {int(path.stem) for path in audio_files}

missing_ids = sorted(expected_ids - downloaded_ids)
extra_ids = sorted(downloaded_ids - expected_ids)

print("===== TRACK ID MATCH CHECK =====")
print("Expected :", len(expected_ids))
print("Found    :", len(downloaded_ids))
print("Missing  :", len(missing_ids))
print("Extra    :", len(extra_ids))

if missing_ids:
    print("\nMissing track IDs:")
    print(missing_ids)

if extra_ids:
    print("\nExtra track IDs:")
    print(extra_ids)

===== TRACK ID MATCH CHECK =====
Expected : 296
Found    : 296
Missing  : 0
Extra    : 0


## 3. 파일 크기 기반 기본 무결성 검사

0 byte 파일이나 지나치게 작은 파일이 있는지 확인한다.

30초 MP3는 보통 수백 KB 이상이므로 여기서는 **100 KB 미만** 파일을 추가 점검 대상으로 표시한다.

In [24]:
# 파일 크기 기반 기본 무결성 검사
file_check = []

for path in audio_files:
    file_check.append(
        {
            "track_id": int(path.stem),
            "path": str(path.relative_to(PROJECT_ROOT)),
            "size_bytes": path.stat().st_size,
        }
    )

file_check = pd.DataFrame(file_check)

zero_byte_count = int((file_check["size_bytes"] == 0).sum())
small_files = file_check[file_check["size_bytes"] < 100_000].copy()

print("0 byte files   :", zero_byte_count)
print("100KB 미만 파일:", len(small_files))

display(file_check["size_bytes"].describe())

if len(small_files):
    display(small_files)

0 byte files   : 0
100KB 미만 파일: 0


count    2.960000e+02
mean     1.021890e+06
std      2.238527e+05
min      2.405400e+05
25%      9.608410e+05
50%      1.159154e+06
75%      1.201366e+06
max      1.203312e+06
Name: size_bytes, dtype: float64

## 4. MP3 디코딩 및 실제 Duration 검사

각 MP3가 실제로 열리는지 확인하고 duration을 측정한다.

우선 `ffprobe`가 설치되어 있으면 이를 사용한다. 설치되어 있지 않으면 `librosa`를 사용하도록 시도한다.

FMA Large는 일반적으로 30초 clip으로 구성되므로 대부분 약 30초가 예상된다. 다만 원곡 자체가 30초보다 짧은 경우 더 짧을 수 있다.

In [6]:
# MP3 디코딩 및 실제 Duration 검사
FFPROBE_AVAILABLE = shutil.which("ffprobe") is not None

print("ffprobe available:", FFPROBE_AVAILABLE)

if not FFPROBE_AVAILABLE:
    try:
        import librosa

        print("librosa available: True")
    except ImportError:
        print("librosa available: False")
        raise RuntimeError(
            "ffprobe 또는 librosa 중 하나가 필요합니다. "
            "Mac에서는 `brew install ffmpeg` 또는 현재 conda 환경에 librosa를 설치하세요."
        )

ffprobe available: True


In [25]:
# MP3 디코딩 및 실제 Duration 검사
def probe_audio_duration(path: Path):
    """Return (ok, duration_sec, error_message)."""
    if FFPROBE_AVAILABLE:
        cmd = [
            "ffprobe",
            "-v",
            "error",
            "-show_entries",
            "format=duration",
            "-of",
            "default=noprint_wrappers=1:nokey=1",
            str(path),
        ]
        result = subprocess.run(
            cmd,
            capture_output=True,
            text=True,
        )

        if result.returncode != 0:
            return False, np.nan, result.stderr.strip()

        try:
            duration = float(result.stdout.strip())
            return True, duration, ""
        except ValueError:
            return False, np.nan, "duration 값을 float로 변환하지 못함"

    else:
        try:
            duration = float(librosa.get_duration(path=str(path)))
            return True, duration, ""
        except Exception as e:
            return False, np.nan, f"{type(e).__name__}: {e}"


audio_validation = []

for i, path in enumerate(audio_files, start=1):
    ok, duration_sec, error = probe_audio_duration(path)

    audio_validation.append(
        {
            "track_id": int(path.stem),
            "path": str(path.relative_to(PROJECT_ROOT)),
            "size_bytes": path.stat().st_size,
            "decode_ok": ok,
            "duration_sec": duration_sec,
            "decode_error": error,
        }
    )

    if i % 50 == 0 or i == len(audio_files):
        print(f"checked: {i}/{len(audio_files)}")

audio_validation = pd.DataFrame(audio_validation)

print("\n===== DECODE CHECK =====")
print("Decode success:", int(audio_validation["decode_ok"].sum()))
print("Decode failed :", int((~audio_validation["decode_ok"]).sum()))

checked: 50/296
checked: 100/296
checked: 150/296
checked: 200/296
checked: 250/296
checked: 296/296

===== DECODE CHECK =====
Decode success: 296
Decode failed : 0


In [26]:
# MP3 디코딩 및 실제 Duration 검사
decode_failures = audio_validation[~audio_validation["decode_ok"]].copy()

if len(decode_failures):
    print("===== DECODE FAILURES =====")
    display(decode_failures)
else:
    print("모든 MP3 파일이 정상적으로 디코딩되었습니다.")

print("\n===== DURATION SUMMARY =====")
display(audio_validation["duration_sec"].describe())

모든 MP3 파일이 정상적으로 디코딩되었습니다.

===== DURATION SUMMARY =====


count    296.000000
mean      29.999641
std        0.012909
min       29.988571
25%       29.988571
50%       29.988571
75%       30.014694
max       30.014694
Name: duration_sec, dtype: float64

## 4.1 총 296개의 real 파일 중 2개가 디코딩 실패
* 실패 원인 : 다운로드 방식 자체는 정상이나 파일의 크기가 너무 작다. 특히 정상적인 294곡의 duration이 30초 정도라는 것으로 미루어보았을 때 해당 파일을 교체해야한다.
* 교체 대상 파일은 복수 후보 즉 전처리 이전 후보군이 2개 이상이었던 곡으로 다음 코드를 활용해 교체를 진행하고자 한다.

In [12]:
# 원격 ZIP 검증에 필요한 remotezip을 현재 커널에 설치한다.
%pip install remotezip

  Using cached remotezip-0.12.6-py3-none-any.whl.metadata (7.3 kB)
Using cached remotezip-0.12.6-py3-none-any.whl (8.6 kB)
Note: you may need to restart the kernel to use updated packages.


In [13]:
from remotezip import RemoteZip

URL = "https://os.unil.cloud.switch.ch/fma/fma_large.zip"

check_ids = [
    148786,  # Loved Ones - 현재 후보
    148802,  # Loved Ones - 대체 후보
    148788,  # Strongbreeze - 현재 후보
    148804,  # Strongbreeze - 대체 후보
]

with RemoteZip(URL) as z:
    for track_id in check_ids:
        folder = f"{track_id // 1000:03d}"
        filename = f"{track_id:06d}.mp3"
        member = f"fma_large/{folder}/{filename}"

        print("=" * 60)
        print("track_id:", track_id)

        try:
            info = z.getinfo(member)

            print("file size      :", info.file_size)
            print("compressed size:", info.compress_size)

        except KeyError:
            print("ZIP에 파일 없음")

track_id: 148786
file size      : 1604
compressed size: 433
track_id: 148802
file size      : 1201147
compressed size: 1192888
track_id: 148788
file size      : 1608
compressed size: 435
track_id: 148804
file size      : 1201151
compressed size: 1196334


## 4.2 대체 후보 2개 파일 다운로드

In [14]:
from remotezip import RemoteZip
from pathlib import Path
import shutil

URL = "https://os.unil.cloud.switch.ch/fma/fma_large.zip"

replacement_ids = [148802, 148804]

output_dir = PROJECT_ROOT / "data/raw/FMA/replacement_check"

with RemoteZip(URL) as z:
    for track_id in replacement_ids:

        folder = f"{track_id // 1000:03d}"
        filename = f"{track_id:06d}.mp3"

        member = f"fma_large/{folder}/{filename}"
        dst = output_dir / folder / filename

        dst.parent.mkdir(parents=True, exist_ok=True)

        with z.open(member) as src, open(dst, "wb") as out:
            shutil.copyfileobj(src, out)

        print(track_id, "->", dst, "|", dst.stat().st_size, "bytes")

148802 -> /Users/seungjae/Desktop/SNU/기계학습 & 딥러닝/project/data/raw/FMA/replacement_check/148/148802.mp3 | 1201147 bytes
148804 -> /Users/seungjae/Desktop/SNU/기계학습 & 딥러닝/project/data/raw/FMA/replacement_check/148/148804.mp3 | 1201151 bytes


## 4-3. 대체 후보 디코딩 + Duration 확인

In [15]:
# 대체 후보 디코딩 + Duration 확인
replacement_results = []

for track_id in replacement_ids:

    folder = f"{track_id // 1000:03d}"
    path = (
        PROJECT_ROOT / "data/raw/FMA/replacement_check" / folder / f"{track_id:06d}.mp3"
    )

    ok, duration_sec, error = probe_audio_duration(path)

    replacement_results.append(
        {
            "track_id": track_id,
            "size_bytes": path.stat().st_size,
            "decode_ok": ok,
            "duration_sec": duration_sec,
            "error": error,
        }
    )

replacement_results = pd.DataFrame(replacement_results)

display(replacement_results)

,track_id,size_bytes,decode_ok,duration_sec,error
0,148802,1201147,True,29.988571,
1,148804,1201151,True,29.988571,


## 4-3 매핑 수정(후보군 2개 교체)

In [17]:
mapping_path = PROJECT_ROOT / "data/metadata/fma_real_mapping.csv"

mapping = pd.read_csv(mapping_path)

replacement_map = {
    148786: 148802,
    148788: 148804,
}

for old_id, new_id in replacement_map.items():
    mask = mapping["track_id"] == old_id

    print(old_id, "->", new_id, "| rows:", mask.sum())

    mapping.loc[mask, "track_id"] = new_id

mapping.to_csv(mapping_path, index=False, encoding="utf-8-sig")

print("Saved:", mapping_path)
print("Rows:", len(mapping))
print("Unique track_id:", mapping["track_id"].nunique())

148786 -> 148802 | rows: 1
148788 -> 148804 | rows: 1
Saved: /Users/seungjae/Desktop/SNU/기계학습 & 딥러닝/project/data/metadata/fma_real_mapping.csv
Rows: 296
Unique track_id: 296


## 4-4 정상 파일 Selected_30s 로 이동 : 아까 교체 확인했던 파일을 실제 real 데이터로 옮기기

In [18]:
import shutil

replacements = {
    148802: 148802,
    148804: 148804,
}

for track_id in replacements:

    folder = f"{track_id // 1000:03d}"
    filename = f"{track_id:06d}.mp3"

    src = PROJECT_ROOT / "data/raw/FMA/replacement_check" / folder / filename

    dst = PROJECT_ROOT / "data/raw/FMA/selected_30s" / folder / filename

    dst.parent.mkdir(parents=True, exist_ok=True)

    shutil.copy2(src, dst)

    print("Copied:", src, "->", dst)

Copied: /Users/seungjae/Desktop/SNU/기계학습 & 딥러닝/project/data/raw/FMA/replacement_check/148/148802.mp3 -> /Users/seungjae/Desktop/SNU/기계학습 & 딥러닝/project/data/raw/FMA/selected_30s/148/148802.mp3
Copied: /Users/seungjae/Desktop/SNU/기계학습 & 딥러닝/project/data/raw/FMA/replacement_check/148/148804.mp3 -> /Users/seungjae/Desktop/SNU/기계학습 & 딥러닝/project/data/raw/FMA/selected_30s/148/148804.mp3


In [20]:
# 이전에 깨진 파일 제거

bad_ids = [148786, 148788]

for track_id in bad_ids:

    folder = f"{track_id // 1000:03d}"
    filename = f"{track_id:06d}.mp3"

    path = PROJECT_ROOT / "data/raw/FMA/selected_30s" / folder / filename

    if path.exists():
        path.unlink()
        print("Removed:", path)

##  다시 1번부터 4번까지 실행!

## 5. Duration 이상값 확인

일반적인 FMA Large clip은 약 30초이다.

다만 FMA 생성 과정에서 원곡 자체가 30초 이하인 경우 전체 길이가 그대로 유지될 수 있으므로, 30초 미만 파일이 존재한다고 해서 바로 오류라고 판단하지 않는다.

여기서는 다음을 별도로 확인한다.

- 0초 이하
- 31초 초과
- 10초 미만

In [27]:
# Duration 이상값 확인
invalid_duration = audio_validation[
    (audio_validation["decode_ok"])
    & (
        (audio_validation["duration_sec"] <= 0)
        | (audio_validation["duration_sec"] > 31)
    )
].copy()

very_short = audio_validation[
    (audio_validation["decode_ok"]) & (audio_validation["duration_sec"] < 10)
].copy()

print("0초 이하 또는 31초 초과:", len(invalid_duration))
print("10초 미만             :", len(very_short))

if len(invalid_duration):
    display(invalid_duration)

if len(very_short):
    print("\n===== 10초 미만 파일 =====")
    display(very_short)

0초 이하 또는 31초 초과: 0
10초 미만             : 0


## 6. 매핑 정보와 오디오 QC 결과 결합

최종적으로 `fma_real_mapping.csv`와 실제 오디오 검증 결과를 `track_id` 기준으로 결합한다.

In [28]:
# 매핑 정보와 오디오 QC 결과 결합
real_validation = mapping.merge(
    audio_validation,
    on="track_id",
    how="left",
    validate="one_to_one",
)

real_validation["file_exists"] = real_validation["path"].notna()
real_validation["size_ok"] = real_validation["size_bytes"].fillna(0) >= 100_000

display(real_validation.head())

,original_audio,genre,track_id,title,artist,genre_top,license,duration,subset,candidate_count,license_allowed,license_fallback,genre_match,path,size_bytes,decode_ok,duration_sec,decode_error,file_exists,size_ok
0,"10,000 People Chanting, ""I'm an Individual"" - ...",Electronic,140932,"10,000 People Chanting, ""I'm an Individual""",Nihilore,Electronic,Creative Commons Attribution,372,medium,1,True,False,True,data/raw/FMA/selected_30s/140/140932.mp3,1200775,True,29.988571,,True,True
1,1984 - Punk Rock Opera,Rock,149410,1984,Punk Rock Opera,Rock,Attribution,200,medium,2,True,False,True,data/raw/FMA/selected_30s/149/149410.mp3,1203312,True,30.014694,,True,True
2,2 (Wasn't There) - Isle of Pine,Rock,66449,2 (Wasn't There),Isle of Pine,Rock,Attribution-NoDerivs 2.5 Canada,108,medium,1,True,False,True,data/raw/FMA/selected_30s/066/066449.mp3,984398,True,30.014694,,True,True
3,2Much (Andy Spinelli & Alex Sánchez House Edit...,Electronic,114244,2Much (Andy Spinelli & Alex Sánchez House Edit),Tentacles,Electronic,Attribution,486,medium,1,True,False,True,data/raw/FMA/selected_30s/114/114244.mp3,1201308,True,29.988571,,True,True
4,3 am West End - statusq,Electronic,112378,3 am West End,statusq,Electronic,Attribution,291,medium,1,True,False,True,data/raw/FMA/selected_30s/112/112378.mp3,600853,True,29.988571,,True,True


## 7. 최종 QC 요약

모든 핵심 항목이 정상인지 최종 확인한다.

In [29]:
# 최종 QC 요약
qc_summary = pd.DataFrame(
    {
        "check": [
            "mapping_rows",
            "unique_track_id",
            "downloaded_mp3",
            "missing_track_id",
            "extra_track_id",
            "zero_byte_files",
            "under_100kb_files",
            "decode_success",
            "decode_failed",
            "duration_outside_0_to_31",
            "under_10sec_files",
        ],
        "value": [
            len(mapping),
            mapping["track_id"].nunique(),
            len(audio_files),
            len(missing_ids),
            len(extra_ids),
            zero_byte_count,
            len(small_files),
            int(audio_validation["decode_ok"].sum()),
            int((~audio_validation["decode_ok"]).sum()),
            len(invalid_duration),
            len(very_short),
        ],
    }
)

display(qc_summary)

all_core_checks_pass = (
    len(mapping) == 296
    and mapping["track_id"].nunique() == 296
    and len(audio_files) == 296
    and len(missing_ids) == 0
    and len(extra_ids) == 0
    and zero_byte_count == 0
    and int((~audio_validation["decode_ok"]).sum()) == 0
)

print("===== FINAL RESULT =====")
print("Core QC PASS:", all_core_checks_pass)

,check,value
0,mapping_rows,296
1,unique_track_id,296
2,downloaded_mp3,296
3,missing_track_id,0
4,extra_track_id,0
5,zero_byte_files,0
6,under_100kb_files,0
7,decode_success,296
8,decode_failed,0
9,duration_outside_0_to_31,0


===== FINAL RESULT =====
Core QC PASS: True


## 8. 검증 결과 저장

곡별 QC 결과를 CSV로 저장한다. 이후 `master_manifest.csv` 생성 시 REAL 데이터의 검증 정보로 사용할 수 있다.

In [30]:
real_validation.to_csv(
    REPORT_PATH,
    index=False,
    encoding="utf-8-sig",
)

print("Saved:", REPORT_PATH)
print("Rows :", len(real_validation))

Saved: /Users/seungjae/Desktop/SNU/기계학습 & 딥러닝/project/data/metadata/fma_real_audio_validation.csv
Rows : 296


## 이어지는 기록

검증을 통과한 REAL 296곡을 04번에서 정제한 FAKE와 합친다.

## REAL 오디오 검사

- 최종 FMA REAL **296곡**을 모두 검증했다.
- 파일 누락, 비정상 크기, 디코딩 실패, 10초 미만 오디오는 모두 **0개**다.
- 교체가 필요했던 track mapping과 실제 파일 반영을 완료했다.
- 검증 결과를 `data/metadata/fma_real_audio_validation.csv`에 296행으로 저장했다.